In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/bisusiddharth/edda-d3-rfmid-clean/D3_clean/samples_per_class.png
/kaggle/input/datasets/bisusiddharth/edda-d3-rfmid-clean/D3_clean/pipeline_funnel.png
/kaggle/input/datasets/bisusiddharth/edda-d3-rfmid-clean/D3_clean/rejected_log.csv
/kaggle/input/datasets/bisusiddharth/edda-d3-rfmid-clean/D3_clean/disease_risk_split.png
/kaggle/input/datasets/bisusiddharth/edda-d3-rfmid-clean/D3_clean/pipeline_stages_example.png
/kaggle/input/datasets/bisusiddharth/edda-d3-rfmid-clean/D3_clean/qa_verification_report.csv
/kaggle/input/datasets/bisusiddharth/edda-d3-rfmid-clean/D3_clean/class_distribution.png
/kaggle/input/datasets/bisusiddharth/edda-d3-rfmid-clean/D3_clean/manifest.csv
/kaggle/input/datasets/bisusiddharth/edda-d3-rfmid-clean/D3_clean/images/d3_train_85.jpg
/kaggle/input/datasets/bisusiddharth/edda-d3-rfmid-clean/D3_clean/images/d3_val_333.jpg
/kaggle/input/datasets/bisusiddharth/edda-d3-rfmid-clean/D3_clean/images/d3_test_138.jpg
/kaggle/input/datasets/bisusiddha

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import f1_score
from torch.cuda.amp import autocast, GradScaler
import os, gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

UNIFIED_LABELS = ["Normal", "DR", "Glaucoma", "AMD", "Myopia"]
SPLIT_PATH = "/kaggle/input/notebooks/bisusiddharth/edda-train-val-test-split/splits/split_D123.csv"

df = pd.read_csv(SPLIT_PATH)
train_df = df[df['split']=='train'].reset_index(drop=True)
val_df   = df[df['split']=='val'].reset_index(drop=True)
test_df  = df[df['split']=='test'].reset_index(drop=True)

class_counts = train_df[UNIFIED_LABELS].sum()
total = len(train_df)
class_weights = torch.tensor([total/(len(UNIFIED_LABELS)*c) for c in class_counts], dtype=torch.float32).to(device)
print("Class weights:", dict(zip(UNIFIED_LABELS, class_weights.cpu().numpy())))

Using: cuda
Class weights: {'Normal': np.float32(0.42400864), 'DR': np.float32(0.6313473), 'Glaucoma': np.float32(1.7529061), 'AMD': np.float32(2.9405), 'Myopia': np.float32(3.5215569)}


In [3]:
IMG_SIZE = 300
BATCH_SIZE = 16

class EyeDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        img = self.transform(img)
        labels = torch.tensor(row[UNIFIED_LABELS].values.astype(np.float32))
        return img, labels

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.25, contrast=0.25),
    transforms.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.9,1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

train_loader = DataLoader(EyeDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(EyeDataset(val_df, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(EyeDataset(test_df, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [4]:
def build_model_full_finetune(arch_name, num_classes=5):
    if arch_name == "vgg16":
        model = models.vgg16(weights="IMAGENET1K_V1")
        model.classifier[6] = nn.Linear(4096, num_classes)
    elif arch_name == "resnet50":
        model = models.resnet50(weights="IMAGENET1K_V2")
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif arch_name == "efficientnet_b3":
        model = models.efficientnet_b3(weights="IMAGENET1K_V1")
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(device)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        focal = (1 - pt) ** self.gamma * bce
        if self.alpha is not None:
            focal = focal * self.alpha
        return focal.mean()

def get_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            with autocast():
                probs = torch.sigmoid(model(imgs)).float().cpu().numpy()
            all_probs.append(probs)
            all_labels.append(labels.numpy())
    return np.vstack(all_probs), np.vstack(all_labels)

def evaluate_f1(model, loader, thresholds=None):
    probs, labels = get_probs(model, loader)
    if thresholds is None:
        thresholds = [0.5]*len(UNIFIED_LABELS)
    preds = np.zeros_like(probs)
    for i, t in enumerate(thresholds):
        preds[:,i] = (probs[:,i] > t).astype(int)
    return f1_score(labels, preds, average='macro', zero_division=0)

def find_best_thresholds(model, loader):
    probs, labels = get_probs(model, loader)
    best_thresholds = []
    for i in range(len(UNIFIED_LABELS)):
        best_f1, best_t = 0, 0.5
        for t in np.arange(0.05, 0.95, 0.02):
            f1 = f1_score(labels[:,i], (probs[:,i] > t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresholds.append(best_t)
    return best_thresholds

def full_report(model, loader, model_name, thresholds):
    probs, labels = get_probs(model, loader)
    preds = np.zeros_like(probs)
    for i, t in enumerate(thresholds):
        preds[:,i] = (probs[:,i] > t).astype(int)
    results = {"Model": model_name, "Macro-F1": f1_score(labels, preds, average='macro', zero_division=0)}
    per_class_f1 = f1_score(labels, preds, average=None, zero_division=0)
    for cls, f1 in zip(UNIFIED_LABELS, per_class_f1):
        results[f"{cls}_F1"] = f1
    return results

In [5]:
def train_model_full_amp(model, name, epochs=30, patience=7):
    criterion = FocalLoss(alpha=class_weights, gamma=2.0)

    if hasattr(model, 'classifier'):
        head_params = list(model.classifier.parameters())
        head_param_ids = set(id(p) for p in head_params)
        backbone_params = [p for p in model.parameters() if id(p) not in head_param_ids]
    else:
        head_params = list(model.fc.parameters())
        head_param_ids = set(id(p) for p in head_params)
        backbone_params = [p for p in model.parameters() if id(p) not in head_param_ids]

    optimizer = torch.optim.Adam([
        {"params": backbone_params, "lr": 1e-5},
        {"params": head_params, "lr": 1e-4}
    ])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    scaler = GradScaler()

    best_val_f1 = 0
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            with autocast():
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item()

        val_f1 = evaluate_f1(model, val_loader)
        scheduler.step(val_f1)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"[{name}] Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Macro-F1: {val_f1:.4f} | LR: {current_lr:.6f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), f"{name}_full_best.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[{name}] Early stopping at epoch {epoch+1} (best val F1: {best_val_f1:.4f})")
                break

    return model

In [6]:
results_table = []

for arch in ["vgg16", "resnet50", "efficientnet_b3"]:
    print(f"\n{'='*60}\nFull fine-tuning: {arch}\n{'='*60}")

    torch.cuda.empty_cache()
    gc.collect()

    model = build_model_full_finetune(arch)
    model = train_model_full_amp(model, arch, epochs=30, patience=7)
    model.load_state_dict(torch.load(f"{arch}_full_best.pth"))

    best_thresholds = find_best_thresholds(model, val_loader)
    print(f"{arch} best thresholds:", dict(zip(UNIFIED_LABELS, best_thresholds)))

    test_results = full_report(model, test_loader, arch, best_thresholds)
    results_table.append(test_results)
    print(f"{arch} TEST results:", test_results)

    del model
    torch.cuda.empty_cache()
    gc.collect()

results_df = pd.DataFrame(results_table)
print(results_df)
results_df.to_csv("/kaggle/working/baseline_results_full_finetune.csv", index=False)


Full fine-tuning: vgg16
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 197MB/s]
/tmp/ipykernel_23/1341230623.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_23/1341230623.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_23/1703190003.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[vgg16] Epoch 1/30 | Loss: 0.1164 | Val Macro-F1: 0.5012 | LR: 0.000010


/tmp/ipykernel_23/1341230623.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_23/1703190003.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[vgg16] Epoch 2/30 | Loss: 0.0886 | Val Macro-F1: 0.6665 | LR: 0.000010


/tmp/ipykernel_23/1341230623.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_23/1703190003.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


... (183 outputs left out)

/tmp/ipykernel_23/1703190003.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


efficientnet_b3 best thresholds: {'Normal': np.float64(0.4700000000000001), 'DR': np.float64(0.4100000000000001), 'Glaucoma': np.float64(0.5500000000000002), 'AMD': np.float64(0.4700000000000001), 'Myopia': np.float64(0.49000000000000005)}


/tmp/ipykernel_23/1703190003.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


efficientnet_b3 TEST results: {'Model': 'efficientnet_b3', 'Macro-F1': 0.7849762167942254, 'Normal_F1': np.float64(0.8151515151515152), 'DR_F1': np.float64(0.7493975903614458), 'Glaucoma_F1': np.float64(0.7137809187279152), 'AMD_F1': np.float64(0.7398843930635838), 'Myopia_F1': np.float64(0.9066666666666666)}
             Model  Macro-F1  Normal_F1     DR_F1  Glaucoma_F1    AMD_F1  \
0            vgg16  0.797952   0.823795  0.727506     0.745387  0.771084   
1         resnet50  0.803782   0.841622  0.760108     0.788927  0.763006   
2  efficientnet_b3  0.784976   0.815152  0.749398     0.713781  0.739884   

   Myopia_F1  
0   0.921986  
1   0.865248  
2   0.906667  
